In [1]:
pip install -U transformers accelerate datasets evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.4.1
    Uninstalling datasets-4.4.1:
      Successfully uninstalled datasets-4.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import json
import re
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import gc

print("Imports successful!")

Imports successful!


In [2]:
print("Loading OpenBioLLM for diagnosis generation...")

# Load OpenBioLLM (for diagnosis generation)
model_id = "aaditya/Llama3-OpenBioLLM-8B"
bio_tokenizer = AutoTokenizer.from_pretrained(model_id)

# Handle missing PAD token (common for LLaMA models)
if bio_tokenizer.pad_token_id is None:
    bio_tokenizer.pad_token = bio_tokenizer.eos_token

bio_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
bio_model.eval()

print("✓ OpenBioLLM loaded successfully")

print("\nLoading SapBERT for semantic embeddings...")

# Load SapBERT (for semantic embeddings - reranking)
sapbert_tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
sapbert_model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
sapbert_model.eval()

print("✓ SapBERT loaded successfully")
print("\n" + "="*60)
print("ALL MODELS LOADED SUCCESSFULLY")
print("="*60 + "\n")

Loading OpenBioLLM for diagnosis generation...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

✓ SapBERT loaded successfully

ALL MODELS LOADED SUCCESSFULLY



In [1]:
from datasets import load_dataset

pubmed = load_dataset(
    "MedRAG/pubmed",
    split="train",
    streaming=True
)

# iterate instead of indexing
for i, example in enumerate(pubmed):
    print(example)
    if i == 5:
        break


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1166 [00:00<?, ?it/s]

{'id': 'pubmed23n0001_0', 'title': "[Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol (author's transl)].", 'content': '(--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH-value. The proteolytic activity of pepsin is reduced by 50 percent through addition of bisabolol in the ratio of 1/0.5. The antipeptic action of bisabolol only occurs in case of direct contact. In case of a previous contact with the substrate, the inhibiting effect is lost.', 'contents': "[Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol (author's transl)]. (--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH-value. The proteolytic activity of pepsin is reduced by 50 percent through addition of bisabolol in the ratio of 1/0.5. The antipeptic act

In [9]:
pip install -q faiss-cpu


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 71.8 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
from datasets import load_dataset
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


In [15]:
def token_chunk_text(
    text,
    tokenizer,
    max_tokens=510,   # <= model max length
    overlap=50
):
    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False
    )["input_ids"]

    chunks = []
    stride = max_tokens - overlap

    for i in range(0, len(token_ids), stride):
        chunk_ids = token_ids[i:i + max_tokens]
        chunks.append(tokenizer.decode(chunk_ids))

    return chunks


In [35]:
embed_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"  # Kaggle GPU
)

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-base-en-v1.5"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [36]:
dim = 768
index = faiss.IndexFlatIP(dim)


In [39]:
BATCH_SIZE = 64
MAX_DOCS = 25000

text_buffer = []
index = faiss.IndexFlatIP(768)
corpus_chunks = []

for i, item in enumerate(pubmed):
    if i >= MAX_DOCS:
        break

    chunks = token_chunk_text(
        item["contents"],
        tokenizer,
        max_tokens=510,
        overlap=50
    )

    for chunk in chunks:
        text_buffer.append(chunk)

        if len(text_buffer) == BATCH_SIZE:
            # 1️⃣ Encode first
            embeddings = embed_model.encode(
                text_buffer,
                batch_size=BATCH_SIZE,
                normalize_embeddings=True,
                show_progress_bar=False
            )

            embeddings = np.asarray(embeddings, dtype="float32")

            index.add(embeddings)

            corpus_chunks.extend(text_buffer)

            text_buffer.clear()

# Flush remainder safely
if text_buffer:
    embeddings = embed_model.encode(
        text_buffer,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True
    )

    embeddings = np.asarray(embeddings, dtype="float32")
    index.add(embeddings)
    corpus_chunks.extend(text_buffer)

assert index.ntotal == len(corpus_chunks)


In [40]:
import faiss

faiss.write_index(index, "pubmed_25k_bge.faiss")


In [41]:
import pickle

with open("pubmed_25k_bge_chunks.pkl", "wb") as f:
    pickle.dump(corpus_chunks, f)


In [42]:
assert index.ntotal == len(corpus_chunks)
print("Index and corpus are aligned:", index.ntotal)


Index and corpus are aligned: 26318


In [43]:
metadata = {
    "model": "BAAI/bge-base-en-v1.5",
    "docs_indexed": 26318,
    "chunk_tokens": 510,
    "overlap": 50,
    "index_type": "IndexFlatIP"
}

with open("pubmed_25k_bge_meta.pkl", "wb") as f:
    pickle.dump(metadata, f)


In [44]:
sample_query = """
A 45-year-old male presents with persistent fever, night sweats,
unintentional weight loss, and enlarged cervical lymph nodes.
Laboratory tests show elevated LDH levels.
"""


In [45]:
query = "Represent this query for retrieving relevant documents: " + sample_query

query_emb = embed_model.encode(
    [query],
    normalize_embeddings=True
)


In [46]:
k = 5  # top-5 retrieval

D, I = index.search(
    np.asarray(query_emb, dtype="float32"),
    k
)


In [47]:
retrieved_texts = [corpus_chunks[i] for i in I[0]]

for rank, (score, text) in enumerate(zip(D[0], retrieved_texts), start=1):
    print(f"\nRank {rank} | Score: {score:.4f}")
    print(text[:600])



Rank 1 | Score: 0.7062
occurrence and natural history of chronic lymphocytic thyroiditis in childhood. in a six - year survey of 5, 179 school children in arizona, utah, and nevada 62 cases of chronic lymphocytic thyroiditis were identified giving a prevalence of 1. 2 %. thyroids were enlarged in 85 %, firm in 60 %, and had an irregular or lobulated surface in 75 %. antibodies to thyroglobulin were demonstrable in the serum at some time during the course of the disease in 76 % by the tanned red blood cell technique and in 93 % by radioimmunoassay. serum tsh concentrations were elevated in seven of 15 subjects. many 

Rank 2 | Score: 0.7049
elevated levels of immunoglobulin e in the acute febrile mucocutaneous lymph node syndrome. mucocutaneous lymph node syndrome ( mcls ) is a newly recognized disease characterized by fever persisting for more than 5 days, an erythematous skin eruption, conjunctival congestion, dry red fissured lips, reddened tongue, palms, and soles, nonpurulent lymp

In [48]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cuda"
)


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [49]:
pairs = [
    (sample_query, text)
    for text in retrieved_texts
]

scores = reranker.predict(pairs)

reranked = sorted(
    zip(scores, retrieved_texts),
    key=lambda x: x[0],
    reverse=True
)


In [50]:
for rank, (score, text) in enumerate(reranked, 1):
    print(f"\nReranked {rank} | Score: {score:.4f}")
    print(text[:600])



Reranked 1 | Score: 0.0911
[ purification and some properties of isozymes 1 and 5 of lactate dehydrogenase from fox heart and skeletal muscles ]. an improved method is described for the isolation of isozymes 1 and 5 of lactate dehydrogenase ( ldh ) from heart and skeletal muscles of foxes. the method includes salt fractionation with ammonium sulphate, chromatography on deae - and cm - celluloses and affinity chromatography on amp - sepharose. the preparations of ldh isozymes 1 and 5 turned out to be homogeneous both in 7, 5 % polyacrylamide gel electrophoresis and under immunodiffusion analysis. it is shown that the ph 

Reranked 2 | Score: 0.0646
elevated levels of immunoglobulin e in the acute febrile mucocutaneous lymph node syndrome. mucocutaneous lymph node syndrome ( mcls ) is a newly recognized disease characterized by fever persisting for more than 5 days, an erythematous skin eruption, conjunctival congestion, dry red fissured lips, reddened tongue, palms, and soles, nonpurul

In [51]:
CLINICAL_TERMS = [
    "patient", "diagnosis", "case", "symptoms",
    "presented with", "clinical", "treated", "prognosis"
]

def is_clinical(text):
    t = text.lower()
    return any(term in t for term in CLINICAL_TERMS)


In [52]:
query = (
    "Represent this query for retrieving relevant clinical case reports "
    "and diagnostic literature: "
    + sample_query
)


In [53]:
query = (
    "Represent this query for retrieving relevant clinical diagnosis "
    "and disease case reports: "
    + sample_query
)

query_emb = embed_model.encode([query], normalize_embeddings=True)
D, I = index.search(np.asarray(query_emb, dtype="float32"), k=20)

retrieved = [corpus_chunks[i] for i in I[0]]


In [58]:
query_emb = embed_model.encode([query], normalize_embeddings=True)
D, I = index.search(np.asarray(query_emb, dtype="float32"), k=20)

retrieved = [corpus_chunks[i] for i in I[0]]


In [59]:
for rank, idx in enumerate(I[0], start=1):
    score = D[0][rank - 1]
    text = corpus_chunks[idx]

    print(f"\nRank {rank} | Score: {score:.4f}")
    print("-" * 80)
    print(text[:800])  # preview first 800 chars



Rank 1 | Score: 0.7266
--------------------------------------------------------------------------------
elevated levels of immunoglobulin e in the acute febrile mucocutaneous lymph node syndrome. mucocutaneous lymph node syndrome ( mcls ) is a newly recognized disease characterized by fever persisting for more than 5 days, an erythematous skin eruption, conjunctival congestion, dry red fissured lips, reddened tongue, palms, and soles, nonpurulent lymphadenopathy, and sometines diarrhea, arthralgia, and aseptic meningitis. additional features may include carditis, pericarditis, aneurysmal dilation and thrombosis of coronary arteries, and sudden death. there is a striking similarity of fatal cases to infantile polyarteritis nodosa, a disease recently reported to be associated with elevated levels of serium ige. indeed, it is likely that mcls represents a disease which can progress to polyarter

Rank 2 | Score: 0.7174
----------------------------------------------------------------------

In [55]:
CLINICAL_TERMS = [
    "patient", "case", "presented with", "diagnosis",
    "symptom", "clinical", "treated", "prognosis",
    "adult", "male", "female", "years old"
]

def is_clinical(text):
    t = text.lower()
    return any(term in t for term in CLINICAL_TERMS)


In [56]:
for i, item in enumerate(pubmed):
    if i >= MAX_DOCS:
        break

    if not is_clinical(item["contents"]):
        continue   # 🔑 this removes enzyme/animal papers

    chunks = token_chunk_text(
        item["contents"],
        tokenizer,
        max_tokens=510,
        overlap=50
    )
    ...


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 166264d4-306f-4ffa-86d5-2f766387ac47)')' thrown while requesting GET https://huggingface.co/datasets/MedRAG/pubmed/resolve/33da3593d5756bc04c8909f170003c0b14197957/chunk/pubmed23n0002.jsonl
Retrying in 1s [Retry 1/5].


In [57]:
query = (
    "Represent this query for retrieving relevant adult clinical case reports, "
    "differential diagnosis discussions, and disease presentations involving "
    "fever, night sweats, weight loss, lymphadenopathy, and elevated LDH: "
    + sample_query
)


In [32]:
query_emb = embed_model.encode(
    test_cases["symptoms_text"],
    normalize_embeddings=True
)

D, I = index.search(np.asarray(query_emb, dtype="float32"), k=10)

retrieved_texts_per_query = [
    [corpus_chunks[i] for i in row]
    for row in I
]


IndexError: list index out of range

In [33]:
import faiss
import pickle

# Save FAISS index
faiss.write_index(index, "pubmed_index.faiss")

# Save text chunks (ID → text mapping)
with open("pubmed_chunks.pkl", "wb") as f:
    pickle.dump(corpus_chunks, f)


In [28]:
test_cases

Dataset({
    features: ['case_id', 'symptoms_text', 'ground_truth'],
    num_rows: 100
})